In [2]:
import os
import shutil
import pandas as pd

# -----------------------------
# paths
# -----------------------------

ROOT = "/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet"

AUDIO_TRAIN = os.path.join(ROOT, "musicnet", "train_data")
AUDIO_TEST = os.path.join(ROOT, "musicnet", "test_data")

LABEL_TRAIN = os.path.join(ROOT, "musicnet", "train_labels")
LABEL_TEST = os.path.join(ROOT, "musicnet", "test_labels")

MIDI_ROOT = os.path.join(ROOT, "musicnet_midis")

META_FILE = os.path.join(ROOT, "musicnet_metadata.csv")

OUTPUT = "MusicNet_solo"

# -----------------------------
# create output folders
# -----------------------------

audio_out = os.path.join(OUTPUT, "audio")
label_out = os.path.join(OUTPUT, "labels")
midi_out = os.path.join(OUTPUT, "midi")

os.makedirs(audio_out, exist_ok=True)
os.makedirs(label_out, exist_ok=True)
os.makedirs(midi_out, exist_ok=True)

# -----------------------------
# load metadata
# -----------------------------

meta = pd.read_csv(META_FILE)

target_ensembles = [
    "solo violin",
    "solo cello",
    "solo flute"
]

filtered = meta[meta["ensemble"].str.lower().isin(target_ensembles)]

ids = filtered["id"].astype(str).tolist()

print(f"{len(ids)} pieces selected")

# -----------------------------
# copy audio
# -----------------------------

def copy_audio(src_folder):
    for f in os.listdir(src_folder):

        if not f.endswith(".wav"):
            continue

        file_id = f.replace(".wav", "")

        if file_id in ids:

            shutil.copy(
                os.path.join(src_folder, f),
                os.path.join(audio_out, f)
            )

copy_audio(AUDIO_TRAIN)
copy_audio(AUDIO_TEST)

# -----------------------------
# copy labels
# -----------------------------

def copy_labels(src_folder):

    for f in os.listdir(src_folder):

        if not f.endswith(".csv"):
            continue

        file_id = f.replace(".csv", "")

        if file_id in ids:

            shutil.copy(
                os.path.join(src_folder, f),
                os.path.join(label_out, f)
            )

copy_labels(LABEL_TRAIN)
copy_labels(LABEL_TEST)

# -----------------------------
# save filtered metadata
# -----------------------------

filtered.to_csv(os.path.join(OUTPUT, "metadata.csv"), index=False)

print("Dataset reorganization complete.")

24 pieces selected
Dataset reorganization complete.


In [3]:
ROOT = "/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet"
MIDI_ROOT = os.path.join(ROOT, "musicnet_midis/Bach")
META = os.path.join("MusicNet_solo", "metadata.csv")

MIDI_OUT = "MusicNet_solo/midi"
os.makedirs(MIDI_OUT, exist_ok=True)

# Load selected IDs
meta = pd.read_csv(META)
ids = set(meta["id"].astype(str))

print("Target IDs:", len(ids))

# Build MIDI lookup
midi_lookup = {}

for root, dirs, files in os.walk(MIDI_ROOT):
    for f in files:
        if f.endswith(".mid") or f.endswith(".midi"):

            for id in ids:
                if id in f:
                    midi_lookup[id] = os.path.join(root, f)

print("Total MIDI files found:", len(midi_lookup))
print(midi_lookup)

# Copy matching files
copied = 0

for file_id in ids:

    if file_id in midi_lookup:

        src = midi_lookup[file_id]
        dst = os.path.join(MIDI_OUT, f"{file_id}.mid")

        shutil.copy(src, dst)
        copied += 1

print("MIDI files copied:", copied)

Target IDs: 24
Total MIDI files found: 24
{'2659': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2659_vs2_6.mid', '2298': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2298_cs4-6gig.mid', '2243': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2243_vs1_3.mid', '2297': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2297_cs4-5bou.mid', '2219': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2219_cs3-3cou.mid', '2186': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2186_vs6_1.mid', '2244': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2244_vs1_4.mid', '2220': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2220_cs3-4sar.mid', '2242': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_midis/Bach/2242_vs1_2.mid', '2294': '/Users/jocelynkavanagh/Documents/MUSI2526/MusicNet/musicnet_mid

In [4]:
import os
import pandas as pd
import numpy as np
import librosa

LABEL_DIR = "MusicNet_solo/labels"
AUDIO_DIR = "MusicNet_solo/audio"

OUTPUT = "MusicNet_solo/pitch_contour_full.csv"

FRAME_STEP = 0.002902 # around 2.9 ms for 128 sample frames for 44.1kHz


rows = []

label_files = [f for f in os.listdir(LABEL_DIR) if f.endswith(".csv")]

for f in label_files:

    file_id = f.replace(".csv","")

    audio_path = os.path.join(AUDIO_DIR, file_id + ".wav")

    if not os.path.exists(audio_path):
        continue

    print("Processing", file_id)

    # get audio duration
    y, sr = librosa.load(audio_path, sr=None)
    duration = len(y) / sr

    n_frames = int(np.ceil(duration / FRAME_STEP))

    times = np.arange(n_frames) * FRAME_STEP

    pitch_contour = np.zeros(n_frames)

    freq_contour = np.zeros(n_frames)

    # load note labels
    labels = pd.read_csv(os.path.join(LABEL_DIR, f))

    for _, note in labels.iterrows():

        start = note["start_beat"]
        end = note["end_beat"]
        pitch = int(note["note"])

        start_frame = int(start / FRAME_STEP)
        end_frame = int(end / FRAME_STEP)

        pitch_contour[start_frame:end_frame] = pitch

        freq = 440 * 2 ** ((pitch - 69) / 12)
        freq_contour[start_frame:end_frame] = freq

    # save rows
    for i, t in enumerate(times):

        rows.append([
            file_id,
            i,
            round(t,3),
            int(pitch_contour[i]),
            float(freq_contour[i])
        ])


df = pd.DataFrame(
    rows,
    columns=["id","frame","time","midi_pitch", "freq"]
)

df.to_csv(OUTPUT, index=False)

print("Saved:", OUTPUT)

Processing 2296
Processing 2241
Processing 2297
Processing 2295
Processing 2242
Processing 2243
Processing 2294
Processing 2293
Processing 2244
Processing 2222
Processing 2221
Processing 2220
Processing 2218
Processing 2191
Processing 2219
Processing 2186
Processing 2203
Processing 2217
Processing 2202
Processing 2204
Processing 2659
Processing 2288
Processing 2289
Processing 2298
Saved: MusicNet_solo/pitch_contour_full.csv


In [5]:
import os
import pandas as pd
import numpy as np
import librosa

LABEL_DIR = "MusicNet_solo/labels"
AUDIO_DIR = "MusicNet_solo/audio"

OUTPUT_DIR = "MusicNet_solo/pitch_contours"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FRAME_STEP = 0.002902 # around 2.9 ms for 128 sample frames for 44.1kHz


label_files = [f for f in os.listdir(LABEL_DIR) if f.endswith(".csv")]

for f in label_files:

    file_id = f.replace(".csv","")

    audio_path = os.path.join(AUDIO_DIR, file_id + ".wav")
    label_path = os.path.join(LABEL_DIR, f)

    if not os.path.exists(audio_path):
        continue

    print("Processing", file_id)

    y, sr = librosa.load(audio_path, sr=None)
    duration = len(y) / sr

    n_frames = int(np.ceil(duration / FRAME_STEP))

    times = np.arange(n_frames) * FRAME_STEP
    samples = np.round(np.arange(n_frames) * FRAME_STEP * sr).astype(int)

    pitch_contour = np.zeros(n_frames)
    freq_contour = np.zeros(n_frames)

    labels = pd.read_csv(label_path)

    for _, note in labels.iterrows():

        start_sample = note["start_time"]
        end_sample = note["end_time"]
        pitch = int(note["note"])

        start_sec = start_sample / sr
        end_sec = end_sample / sr

        start_frame = int(start_sec / FRAME_STEP)
        end_frame = int(end_sec / FRAME_STEP)

        pitch_contour[start_frame:end_frame] = pitch

        freq = 440 * 2 ** ((pitch - 69) / 12)
        freq_contour[start_frame:end_frame] = freq


    contour_df = pd.DataFrame({
        "sample": samples,
        "time": np.round(times,3),
        "midi_pitch": pitch_contour.astype(int),
        "frequency": freq_contour
    })


    output_file = os.path.join(
        OUTPUT_DIR,
        f"{file_id}_pitch.csv"
    )

    contour_df.to_csv(output_file, index=False)

print("Pitch contour files created.")

Processing 2296
Processing 2241
Processing 2297
Processing 2295
Processing 2242
Processing 2243
Processing 2294
Processing 2293
Processing 2244
Processing 2222
Processing 2221
Processing 2220
Processing 2218
Processing 2191
Processing 2219
Processing 2186
Processing 2203
Processing 2217
Processing 2202
Processing 2204
Processing 2659
Processing 2288
Processing 2289
Processing 2298
Pitch contour files created.
